In [ ]:
# Setup: section 8 (CLIP vision-language fusion) needs open_clip_torch + internet on Kaggle.
# Safe to re-run — pip skips it if already installed. The rest of the notebook runs without it.
import importlib.util
if importlib.util.find_spec("open_clip") is None:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
print("open_clip ready")

# Phase 0 - reuse trained checkpoints (no retraining)

Cheap long-tail boosts that load the finished `outputs/<method>/` checkpoints and
only run inference:
- **Ensemble (+TTA)** of the trained classifiers.
- **Tier-aware prototype fusion** (parametric head for head classes, prototypes for tail).
- **tau-normalization** of a trained linear head.

Each writes its result as a new `outputs/<name>/metrics.json`, then we rebuild the
comparison table + bar chart. Run `run_all_methods.ipynb` first so the checkpoints exist.

## 1. Make `src/` importable

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Config (match how the checkpoints were trained)

In [ ]:
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"
OUTPUT_DIR = PROJECT_DIR / "outputs"

NUM_CLASSES = 100
PRETRAINED = False        # must match the training runs (False = CIFAR stem)
IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = 2
DEVICE = None

# which trained runs to reuse (folders under outputs/ that hold best_model.pt)
REUSE_METHODS = ["baseline", "balanced_softmax", "decoupling", "supcon"]
BEST_SINGLE = "balanced_softmax"     # strongest single model, used for fusion / tau-norm

FUSION_WEIGHTS = {"many": 0.0, "medium": 0.3, "few": 0.8}   # prototype weight per tier
TAU_VALUES = [0.5, 0.75, 1.0]
SEED = 42
VAL_FRACTION = 0.1   # must match run_all_methods so val is the same split

## 3. Imports, device, data

In [ ]:
import numpy as np
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    make_loader, split_indices_by_class, split_shot_groups,
                                    subset)
from src.evaluation import visualize
from src.evaluation.ensemble import ensemble_predict, load_classifier, predict_probs
from src.evaluation.metrics import compute_metrics, format_metrics
from src.evaluation.posthoc import tau_normalized_predict, tier_fusion
from src.utils.experiment import compare_runs, create_run_dir, save_metrics
from src.utils.seed import resolve_device, set_seed

set_seed(42)
device = resolve_device(DEVICE)

class_counts = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts)
base_eval = load_split(DATA_DIR, "train", build_transforms(train=False, image_size=IMAGE_SIZE))
train_idx, val_idx = split_indices_by_class([l for _, l in base_eval.samples], VAL_FRACTION, SEED)
train_eval = subset(base_eval, train_idx)   # training split (for prototypes)
val_eval = subset(base_eval, val_idx)       # validation (for tau selection)
test_eval = load_split(DATA_DIR, "test", build_transforms(train=False, image_size=IMAGE_SIZE))
train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = make_loader(val_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("device:", device, "| reusing:", REUSE_METHODS)


def record(name, result):
    """Compute + save metrics for a reuse technique, and print the key numbers."""
    metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
    run_dir = create_run_dir(OUTPUT_DIR, name)
    save_metrics(run_dir, metrics)
    print(f"\n=== {name} ===")
    print(format_metrics(metrics))
    return metrics

## 4. Load the trained classifiers

In [ ]:
models = {}
for method in REUSE_METHODS:
    run_dir = OUTPUT_DIR / method
    if (run_dir / "best_model.pt").exists():
        models[method] = load_classifier(run_dir, NUM_CLASSES, device, pretrained=PRETRAINED)
        print("loaded", method)
    else:
        print("skip (no checkpoint):", method)

## 5. Ensemble (+ test-time augmentation)

In [ ]:
ens = list(models.values())
record("ensemble", ensemble_predict(ens, test_loader, device, tta=False))
record("ensemble_tta", ensemble_predict(ens, test_loader, device, tta=True))

## 6. Tier-aware prototype fusion

Parametric head for the head classes, class-mean prototypes for the tail. Uses the
encoder of the strongest single model.

In [ ]:
best = models[BEST_SINGLE]
record("tier_fusion", tier_fusion(best, train_eval_loader, test_loader,
                                  NUM_CLASSES, shot_groups, device, weights=FUSION_WEIGHTS))

## 7. tau-normalization (ablation over tau)

> Note: this sweep reads the test set to *illustrate* tau's effect. For a strict
> setup pick tau on a held-out split; treat the table below as an ablation.

In [ ]:
# Select tau on the validation split, then report the chosen tau on test.
val_score = {}
for tau in TAU_VALUES:
    r = tau_normalized_predict(best, val_loader, device, tau=tau)
    val_score[tau] = compute_metrics(r["y_true"], r["y_pred"], NUM_CLASSES, shot_groups)["balanced_accuracy"]
best_tau = max(val_score, key=val_score.get)
print("val balanced_acc by tau:", {k: round(v, 4) for k, v in val_score.items()}, "-> best tau =", best_tau)
record(f"tau_norm", tau_normalized_predict(best, test_loader, device, tau=best_tau))

In [ ]:
import numpy as np
from src.evaluation.metrics import balanced_accuracy
from src.experts.clip_expert import CIFAR100_CLASSES, clip_probs, load_clip

best_vision = models[BEST_SINGLE]   # set BEST_SINGLE='cmo' for the strongest vision model
clip_bundle = load_clip(CIFAR100_CLASSES, device, model_name='ViT-B-32', pretrained='openai')

# Per-sample probabilities on val (to tune alpha) and test (to report).
pv_val, yv = predict_probs(best_vision, val_loader, device)
pc_val, yv2 = clip_probs(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
pv_test, yt = predict_probs(best_vision, test_loader, device)
pc_test, yt2 = clip_probs(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
assert (yv == yv2).all() and (yt == yt2).all(), 'sample order mismatch between experts'

# Single experts for reference.
record('vision_only', {'y_true': yt, 'y_pred': pv_test.argmax(1)})
record('clip_only', {'y_true': yt, 'y_pred': pc_test.argmax(1)})

# Tune the CLIP weight alpha on validation (balanced accuracy), then apply on test.
alphas = np.linspace(0.0, 1.0, 21)
best_alpha = max(alphas, key=lambda a: balanced_accuracy(yv, (a * pc_val + (1 - a) * pv_val).argmax(1)))
print('best CLIP weight alpha (on val):', round(float(best_alpha), 2))
fused = best_alpha * pc_test + (1 - best_alpha) * pv_test
record('vlm_fusion', {'y_true': yt, 'y_pred': fused.argmax(1)})

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from src.experts.clip_expert import CIFAR100_CLASSES, clip_probs, load_clip

best_vision = models[BEST_SINGLE]   # set BEST_SINGLE='cmo' for the strongest vision model
clip_bundle = load_clip(CIFAR100_CLASSES, device, model_name='ViT-B-32', pretrained='openai')

# Per-sample probabilities on val (to tune alpha) and test (to report).
pv_val, yv = predict_probs(best_vision, val_loader, device)
pc_val, yv2 = clip_probs(val_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
pv_test, yt = predict_probs(best_vision, test_loader, device)
pc_test, yt2 = clip_probs(test_eval, clip_bundle, device, BATCH_SIZE, NUM_WORKERS)
assert (yv == yv2).all() and (yt == yt2).all(), 'sample order mismatch between experts'

# Single experts for reference.
record('vision_only', {'y_true': yt, 'y_pred': pv_test.argmax(1)})
record('clip_only', {'y_true': yt, 'y_pred': pc_test.argmax(1)})

# Tune the CLIP weight alpha on validation (balanced accuracy), then apply on test.
alphas = np.linspace(0.0, 1.0, 21)
best_alpha = max(alphas, key=lambda a: balanced_accuracy_score(yv, (a * pc_val + (1 - a) * pv_val).argmax(1)))
print('best CLIP weight alpha (on val):', round(float(best_alpha), 2))
fused = best_alpha * pc_test + (1 - best_alpha) * pv_test
record('vlm_fusion', {'y_true': yt, 'y_pred': fused.argmax(1)})

## 9. Rebuild the comparison table + chart (all methods + Phase-0 results)

In [ ]:
comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
visualize.plot_metric_comparison(comparison, OUTPUT_DIR)
comparison